In [37]:
# === PARAMETERS ===
MODEL_TYPE = "CNN" # "MLP" or "CNN"
LOG_FILE = "training_log_decentralized.txt"
STATES_LOG_FILE = "states_log_decentralized.txt"

# Grid
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 2
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09

# MLP Configuration
MLP_HIDDEN_DIM = 512
MLP_NUM_LAYERS = 4

# CNN Configuration
CNN_CONV_CHANNELS = [16, 32]
CNN_HEAD_HIDDEN_DIM = 128
CNN_HEAD_NUM_LAYERS = 2
CNN_KERNEL_SIZE = 3

# Training
LR = 0.0001
BATCH_SIZE = 32
DISCOUNT = 0.0
STOP_THRESHOLD_MAE = 0.01 
SEED = 42
MAX_STEPS = 500_000
EVAL_FREQ = 1000

In [32]:
import sys
import numpy as np
import torch
import random
from tqdm import tqdm
import matplotlib.pyplot as plt

sys.path.append("../") 
from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from tadd_helpers.eval_functions import nearest_policy

from models.value_mlp import ValueMLPDecentralized
from models.value_cnn_new import ValueCNNDecentralized

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [33]:
if MODEL_TYPE == "MLP":
    model = ValueMLPDecentralized(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS
    )
elif MODEL_TYPE == "CNN":
    model = ValueCNNDecentralized(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
        conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
    )
else:
    raise ValueError("Invalid MODEL_TYPE")

params = sum(p.numel() for p in model.policy_net.parameters())
print(f"Initialized {MODEL_TYPE}. Total Trainable Parameters: {params}")

Initialized CNN. Total Trainable Parameters: 25953


In [34]:
def generate_decentralized_test_suite(n=500):
    states_zero, zeros_idx = [], []
    states_pick, pick_idx = [], []
    states_by, by_idx = [], []
    
    def add_noise_apples(s):
        num_noise = np.random.randint(1, 6)
        cnt = 0
        while cnt < num_noise:
            r, c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if s.apples[r, c] == 0 and s.agents[r, c] == 0:
                s.apples[r, c] = 1
                cnt += 1

    # 1. Zeros (Target 0.0)
    while len(states_zero) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        add_noise_apples(s)
        
        # VALIDITY CHECK: No agent can be on an apple.
        # If ANY agent is on an apple, Agent 0 is a Bystander (Reward > 0), not Zero.
        valid = True
        for i in range(NUM_AGENTS):
            if s.apples[tuple(s.agent_position(i))] > 0: valid = False
        
        if valid:
            states_zero.append(s)
            zeros_idx.append(0) 
            
    # 2. Pickers (Target -1.0)
    while len(states_pick) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        add_noise_apples(s)
        
        s.set_agent_position(0, np.array([0,0]))
        s.apples[0,0] = 1
        states_pick.append(s)
        pick_idx.append(0)
        
    # 3. Bystanders (Target 2/(N-1))
    while len(states_by) < n:
        s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
        add_noise_apples(s)
        
        # Place Picker (Agent 1)
        r, c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
        s.set_agent_position(1, np.array([r, c])) 
        s.apples[r, c] = 1
        
        # Place Bystander (Agent 0)
        while True:
            br, bc = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            # Bystander must NOT be on the picker's apple OR any noise apple
            if (br, bc) != (r, c) and s.apples[br, bc] == 0:
                s.set_agent_position(0, np.array([br, bc])) 
                break
                
        states_by.append(s)
        by_idx.append(0)
        
    return (states_zero, zeros_idx), (states_pick, pick_idx), (states_by, by_idx)

(z_s, z_i), (p_s, p_i), (b_s, b_i) = generate_decentralized_test_suite()
print(f"Test Suite: {len(z_s)} Zeros, {len(p_s)} Pickers, {len(b_s)} Bystanders")

Test Suite: 500 Zeros, 500 Pickers, 500 Bystanders


In [ ]:
loss_history = []
max_ape_history = []

# === RE-SEED ===
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# === CONFIGURATION ===
model.discount = 0.0 
s: State = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)

# Log Files
LOG_FILE = "training_log.txt"
STATES_LOG_FILE = "states_log.txt"
BATCH_SIZE = 64 # Increased to ensure mix of 0s and 1s

# Clear/Init log files
with open(LOG_FILE, "a") as f:
    f.write(f"\n=== STARTING DECENTRALIZED TRAINING ({MODEL_TYPE}) - BALANCED ===\n")
    f.write("Step | Loss    | Max%_Z | Avg%_Z | Max%_P | Avg%_P | Max%_B | Avg%_B\n")

with open(STATES_LOG_FILE, "a") as f:
    f.write(f"\n=== STARTING DECENTRALIZED TRAINING ({MODEL_TYPE}) ===\n")

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    
    # --- 1. MOVE (t -> t+0.5) ---
    active_agent_idx = s.get_random_agent_id()
    move_action = nearest_policy(s, active_agent_idx)
    
    old_pos = s.agent_position(active_agent_idx)
    new_pos = np.clip(old_pos + move_action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
    
    s_mid = s.copy()
    s_mid.set_agent_position(active_agent_idx, new_pos)
    
    # === STRATEGY: UNDERSAMPLE ZEROS ===
    # Keep only 10% of moves (Reward 0)
    if random.random() < 0.1: 
        for i in range(NUM_AGENTS):
            model.add_experience(s, s_mid, 0.0, agent_pos=s.agent_position(i))

    # --- 2. PICK (t+0.5 -> t+1) ---
    s_next = s_mid.copy()
    rewards = {i: 0.0 for i in range(NUM_AGENTS)}
    
    is_event = False
    if s_next.apples[tuple(new_pos)] > 0:
        is_event = True 
        s_next.remove_apple_at(new_pos)
        rewards[active_agent_idx] = -1.0
        if NUM_AGENTS > 1:
            bystander_reward = 2.0 / (NUM_AGENTS - 1)
            for i in range(NUM_AGENTS):
                if i != active_agent_idx:
                    rewards[i] = bystander_reward
                    
    spawn_apples(s_next, SPAWN_PROB)
    despawn_apples(s_next, DESPAWN_PROB)
    
    # === STRATEGY: KEEP ALL EVENTS ===
    if is_event:
        # Store 100% of Picks/Bystanders
        for i in range(NUM_AGENTS):
            model.add_experience(s_mid, s_next, rewards[i], agent_pos=s_mid.agent_position(i))
    elif random.random() < 0.1:
        # Store 10% of boring env updates
        for i in range(NUM_AGENTS):
            model.add_experience(s_mid, s_next, 0.0, agent_pos=s_mid.agent_position(i))

    # --- 3. TRAIN ---
    loss = model.train_batch(BATCH_SIZE)
    if loss is not None:
        loss_history.append(loss)
    s = s_next
    
    # --- 4. EVAL ---
    if step % EVAL_FREQ == 0 and step > 0:
        # Zero
        z_preds = np.array([model.get_value(z_s[k], agent_pos=z_s[k].agent_position(z_i[k])) for k in range(len(z_s))])
        abs_z = np.abs(z_preds)
        max_ape_z = np.max(abs_z) * 100
        avg_ape_z = np.mean(abs_z) * 100
        
        # Picker
        p_preds = np.array([model.get_value(p_s[k], agent_pos=p_s[k].agent_position(p_i[k])) for k in range(len(p_s))])
        abs_p = np.abs(p_preds - (-1.0))
        max_ape_p = np.max(abs_p) * 100
        avg_ape_p = np.mean(abs_p) * 100
        
        # Bystander
        if NUM_AGENTS > 1:
            target_b = 2.0 / (NUM_AGENTS - 1)
            b_preds = np.array([model.get_value(b_s[k], agent_pos=b_s[k].agent_position(b_i[k])) for k in range(len(b_s))])
            abs_b = np.abs(b_preds - target_b) / target_b
            max_ape_b = np.max(abs_b) * 100
            avg_ape_b = np.mean(abs_b) * 100
        else:
            max_ape_b, avg_ape_b = 0.0, 0.0
            
        global_max = max(max_ape_z, max_ape_p, max_ape_b)
        
        # Log Metrics
        with open(LOG_FILE, "a") as f:
            f.write(f"{step:<6} | {loss:.5f} | {max_ape_z:6.2f} | {avg_ape_z:6.2f} | {max_ape_p:6.2f} | {avg_ape_p:6.2f} | {max_ape_b:6.2f} | {avg_ape_b:6.2f}\n")
            
        # Log State
        with open(STATES_LOG_FILE, "a") as f:
            f.write(f"\n--- Step {step} ---\n")
            f.write(str(s) + "\n")
        
        pbar.set_description(f"L:{loss:.4f} | Max%:{global_max:.0f}%")
        
        if global_max < 1.0:
            print(f"✅ CONVERGED at Step {step}")
            break

L:0.0000 | Max%:191% | Avg%Z:0.2%:   1%|          | 6000/500000 [00:27<38:00, 216.66it/s]  


KeyboardInterrupt: 